<a href="https://colab.research.google.com/github/wei-wei122/114-2-ProgramingLanguage/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
!pip install -q google-generativeai

In [52]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [68]:
from google.colab import userdata
userdata.get('gemini-1.5-flash2.0')

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini-1.5-flash2.0')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

MODEL_ID = 'gemini-flash-latest'

In [69]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1aVgX-HpCFHXSGXp04HIZ-YlI3UCq1fJtJRryv5sfBco/edit?gid=0#gid=0"
WORKSHEET_NAME = "工作表1"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [70]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [71]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    model = genai.GenerativeModel(MODEL_ID)

    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = model.generate_content(prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [72]:
from datetime import datetime
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：math
請輸入 math 的成績：88
已記錄：日期: 2026-04-01, 科目: math, 成績: 88

請輸入科目（或輸入 'q' 停止）：physics
請輸入 physics 的成績：77
已記錄：日期: 2026-04-01, 科目: physics, 成績: 77

請輸入科目（或輸入 'q' 停止）：q


In [73]:
new_grades

[['2026-04-01', 'math', 88], ['2026-04-01', 'physics', 77]]

In [74]:
import google.generativeai as genai
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6468.63ms


'根據您提供的成績資料（2026-04-01），以下是針對這兩門科目的成績摘要與常見迷思整理：\n\n### 一、成績摘要\n*   **測驗日期**：2026-04-01\n*   **科目表現**：\n    *   **數學**：88分。表現優異，顯示對該單元的核心概念有良好的掌握。\n    *   **物理**：77分。表現尚可，顯示已具備基本理解，但在複雜題型或觀念應用上可能仍有進步空間。\n*   **整體觀察**：兩科皆屬於邏輯與運算類科目，數學表現優於物理，可能代表在純數值運算上較具優勢，而物理涉及的具體情境分析則需多加留意。\n\n---\n\n### 二、常見迷思整理\n\n針對這兩個科目，學生在此分數區段常遇到的迷思如下：\n\n#### 【數學科】\n1.  **公式套用而不理解來源**：有時能算出答案是因為背下了公式，但當題目敘述改變或結合不同觀念時，容易找不到切入點。\n2.  **忽略定義細節**：例如在處理函數、對數或幾何時，常忽略了「定義域」或「前提限制」，導致在陷阱題中出錯。\n3.  **計算過程過於簡略**：在處理多步驟題目時，因跳過步驟而導致正負號或小數位數出錯。\n\n#### 【物理科】\n1.  **單位換算的疏忽**：常在運算過程中忘記統一單位（如公尺與公分、公斤與公克），導致最終結果倍數錯誤。\n2.  **生活直覺與科學定律的衝突**：容易用「直覺」思考（例如：重物下落一定比較快），而忽略了物理定律（如忽略空氣阻力的自由落體）的適用前提。\n3.  **向量與純量的混淆**：在處理力、速度或位移時，常只考慮數值大小而忽略了「方向性」，這是 70 分段位學生最常出現的失誤點。\n4.  **公式適用情境不明**：物理公式多有其特定條件（如：等加速度公式僅適用於加速度固定時），學生有時會不分青紅皂白地帶入公式。\n\n這份整理旨在幫助了解目前的學習狀況，您可以針對上述迷思進行自我檢視，看看是否符合目前的作答情形。'

In [75]:
from google.colab import auth
from google.auth import default
import gspread
from datetime import datetime

def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：math
請輸入 math 的成績：88
已記錄：日期: 2026-04-01, 科目: math, 成績: 88

請輸入科目（或輸入 'q' 停止）：phtsics
請輸入 phtsics 的成績：77
已記錄：日期: 2026-04-01, 科目: phtsics, 成績: 77

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-flash-latest:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 16570.86ms



--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
根據您提供的成績資料（2026-04-01），以下是針對數學與物理兩科的簡要摘要與常見迷思整理：

### 一、 成績摘要報告
*   **測驗日期**：2026-04-01
*   **科目表現**：
    *   **數學**：88分。表現穩定，顯示對該單元的核心概念有相當程度的掌握。
    *   **物理**：77分。表現尚可，但相較於數學，可能在複雜觀念的應用或計算細節上存在些許落差。
*   **整體觀察**：數理兩科表現均在水準之上，建議重點放在縮小物理科的觀念死角，以平衡兩科的發展。

---

### 二、 常見迷思與學習重點整理
針對這兩門科目，學生在該分段成績中常遇到的挑戰與迷思整理如下：

#### 【數學科：80-90分階段常見迷思】
1.  **忽略限制條件**：在解方程或不等式時，常忘記考慮分母不為零、根號內須大於等於零等基本定義域限制。
2.  **公式套用慣性**：看到類似的題型就直接帶入公式，而忽略了題目細微的條件變化（例如：等差與等比數列的定義混淆）。
3.  **計算細節疏漏**：此分數段的學生通常觀念正確，但常在正負號轉換或繁瑣的分數運算中出現非受迫性失誤。

#### 【物理科：70-80分階段常見迷思】
1.  **觀念與定義混淆**：
    *   例如：分不清楚「速度」與「速率」、「位移」與「路徑長」。
    *   在動力學中，常將「合力」與「分力」的概念搞混。
2.  **單位轉換錯誤**：未能將數據統一轉換為標準單位（如：km/h 轉 m/s），導致計算結果雖然邏輯正確但數字錯誤。
3.  **公式適用範圍不明**：物理公式通常有前提條件（例如：等加速度運動公式僅適用於加速度固定的情況），學生有時會不分前提地套用公式。
4.  **向量與標量的忽略**：在處理力學或運動學時，忘記考慮方向性（正負號），這也是物理成績無法往上突破的常見原因。

---

**建議建議：**
物理科可加強「圖表分析」（如 V-t 圖）與「受力分析圖（FBD）」的畫法，這有助於將抽象概念具體化，減少解題時的迷思。
--------

In [76]:
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025